# Plot dbz, plot planview
Loops through all wrfout files and computes 'c' as in Squitieri and Gallus, 2020. Each file is saved per timestamp

In [2]:
import wrf
from wrf import CoordPair, vertcross, get_cartopy, getvar, latlon_coords, to_np, destagger, ll_to_xy
import numpy as np
from netCDF4 import Dataset 
import matplotlib.pyplot as plt
import pyart
import geocat.viz as gv
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import os
import glob
from metpy.units import units
import cartopy.crs as crs
import pandas as pd
import xarray as xr
import metpy 
import metpy.calc as mpcalc
import numpy.ma as ma
import matplotlib.ticker as mticker

import cartopy.feature as cfeature
from metpy.interpolate import cross_section
import pkg_resources


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [3]:
# Set up for saving graphics files
plotsdir = '/glade/u/home/athornton/wrf_visualization/cold_pool_data/dbz_pv_cp/'

# Set directory where wrfout files reside, and list the files for processing.  Set up for a directory with only wrfout files.
os.chdir("/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1")
os.getcwd()

'/glade/campaign/univ/uncs0067/cold_pool_experiments/coldpool1'

In [4]:
datafiles = ["wrfout_d03_2018-08-26_00:00:00",
             "wrfout_d03_2018-08-26_03:00:00",
             "wrfout_d03_2018-08-26_09:00:00",
             "wrfout_d03_2018-08-26_12:00:00",
             "wrfout_d03_2018-08-26_14:30:00",
             "wrfout_d03_2018-08-26_18:00:00",
             "wrfout_d03_2018-08-27_00:00:00",
]
numfiles=len(datafiles)
print(numfiles)


7


In [5]:
input = "wrfout_d03_2018-08-25_23:30:00" 
ncfile = Dataset(input)
slp = getvar(ncfile, "slp")
lats, lons = latlon_coords(slp)
lats_np = to_np(lats)
lons_np = to_np(lons)
cart_proj = get_cartopy(wrfin=ncfile)
print (cart_proj)
proj=crs.PlateCarree()

+proj=merc +a=6370000.0 +b=6370000.0 +nadgrids=@null +lon_0=0.0 +lat_ts=14.828 +k=1 +units=m +no_defs +type=crs


In [6]:
# Slice these arrays so they work with the new dataset
lat = to_np(lats)[:,0]
lon = to_np(lons)[0]

In [ ]:
datafiles

In [10]:
# define plotting function
def create_fig():
    fig = plt.figure(figsize=(12,12))
    plt.rcParams['savefig.dpi'] = 255

    proj=crs.PlateCarree()

    ax = plt.axes(projection=proj)
    ax.set_extent([-26, -4, 1, 24],crs.PlateCarree())
    #ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=.5, edgecolor="black")
    ax.add_feature(cfeature.COASTLINE.with_scale('10m'), linewidth=0.8, edgecolor='black')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=1.0)
    
    #gv.set_titles_and_labels(ax, lefttitle=" 600 hPa Zonal Winds", lefttitlefontsize=18)
    plt.title('Simulated reflectivity', loc='left', weight='bold', fontsize=22)
    plt.title(titletime, loc='right', fontsize=22)
    gl1 = ax.gridlines(crs=proj, linewidth=1, color='darkgray', draw_labels=True)
    gl1.top_labels = False
    gl1.right_labels = False
    gl1.xlocator = mticker.FixedLocator(np.arange(-90,40,4))
    gl1.ylocator = mticker.FixedLocator(np.arange(-2,30,4))
    gl1.xformatter = LongitudeFormatter(degree_symbol='')
    gl1.yformatter = LatitudeFormatter(degree_symbol='')
    gl1.xlabel_style = {'size': 22}
    gl1.ylabel_style = {'size': 22}
    
    #levels = np.arange(0, 15, 0.1) ## c2
    
    ax.plot([lonmin1,lonmax1], [latmin1,latmax1], '-o', color='darkgoldenrod',linewidth=2.)
    ax.plot([lonmin2,lonmax2], [latmin2,latmax2], '-o', color='darkgoldenrod',linewidth=2.)
    ax.plot([lonmin3,lonmax3], [latmin3,latmax3], '-o', color='darkgoldenrod',linewidth=2.)
    # Add dBZ contours
    levels = list(range(17, 80, 3))   # -10 to 75 dBZ in 5 dBZ steps
    
    cf = plt.contourf(lons_np, lats_np, mdbz,
                      levels=levels, extend='max',
                         cmap='HomeyerRainbow', transform=proj)
    plt.tight_layout()
    cb = plt.colorbar(cf, ax=ax, orientation="horizontal", pad=.06, shrink=.6, aspect=30)
    cb.set_label('$Reflectivity$ ($dBZ$)', fontsize=16)
    
    outTPlotName= 'd03_dbz_'+filetime+'.png'
    plt.savefig(plotsdir+outTPlotName)

In [11]:
latmin1 = 8.5
latmax1 = (65.5/5.75)
lonmin1 = -16.5
lonmax1 = -7

latmin2 = (64/6.)
latmax2 = 14.
lonmin2 = -20.
lonmax2 = -12.

## Oceanic MCS
latmin3 = 5.
latmax3 = 10.
lonmin3 = -20.
lonmax3 = -18.

In [12]:
# times = []
# c_list = []
for i in range(4, 5):
    wrf_out_data = xr.open_dataset(datafiles[i])  
    ncfile = Dataset(datafiles[i])
    Time=wrf.extract_times(ncfile, timeidx=0, method='cat', squeeze=True, cache=None, meta=False, do_xtime=False)
    timestr=(str(Time))
    # Set up one time string for plot titles, another for file names
    titletime=(timestr[0:10]+' '+timestr[11:16])
    filetime=(timestr[0:10]+'_'+timestr[11:16])
    print('WRF valid time: ',filetime)
    
    mdbz = getvar(ncfile, "mdbz")

    f = create_fig()
    plt.close()
    

WRF valid time:  2018-08-26_14:30
